# Métodos modernos de recuperación de información

La recuperación *sparse* representa consultas y documentos mediante términos explícitos. Métodos como BM25 son eficientes, interpretables y especialmente fuertes ante nombres propios, códigos y coincidencias exactas. La recuperación *dense* usa embeddings para aproximar significado aunque no haya coincidencia literal.

Ninguna señal domina siempre: «automóvil» y «coche» favorecen la semántica, mientras que un identificador como `ZX-81` favorece la señal léxica. Los sistemas híbridos combinan ambas para aprovechar su complementariedad. Sobre ese primer ranking pueden añadirse etapas de reranking, interacción tardía y representaciones multimodales.

In [1]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 72)
pd.set_option("display.precision", 4)

In [ ]:
corpus = [
    Document(page_content="BM25 pondera la frecuencia de términos y la longitud del documento.", metadata={"id": "D01", "tema": "sparse"}),
    Document(page_content="Los embeddings densos representan el significado semántico en vectores.", metadata={"id": "D02", "tema": "dense"}),
    Document(page_content="La búsqueda híbrida combina señales léxicas y semánticas.", metadata={"id": "D03", "tema": "hybrid"}),
    Document(page_content="Reciprocal Rank Fusion combina rankings sin calibrar sus puntuaciones.", metadata={"id": "D04", "tema": "fusion"}),
    Document(page_content="Un cross-encoder evalúa conjuntamente una consulta y un documento.", metadata={"id": "D05", "tema": "reranking"}),
    Document(page_content="ColBERT conserva un vector por token y aplica interacción tardía MaxSim.", metadata={"id": "D06", "tema": "late interaction"}),
    Document(page_content="La búsqueda multimodal relaciona texto, imágenes, audio y vídeo.", metadata={"id": "D07", "tema": "multimodal"}),
    Document(page_content="El índice invertido permite recuperar rápidamente documentos por palabras.", metadata={"id": "D08", "tema": "sparse"}),
    Document(page_content="El modelo de vehículo ZX-81 tiene autonomía eléctrica de 420 kilómetros.", metadata={"id": "D09", "tema": "producto"}),
    Document(page_content="Un automóvil eléctrico puede recorrer largas distancias sin emisiones locales.", metadata={"id": "D10", "tema": "producto"}),
]

queries = {
    "Q1": "cómo combinar coincidencia exacta y significado",
    "Q2": "coche eléctrico con gran autonomía",
    "Q3": "especificaciones ZX-81",
    "Q4": "comparar cada token de la consulta con tokens del documento",
}

corpus_df = pd.DataFrame([
    {"id": doc.metadata["id"], "tema": doc.metadata["tema"], "texto": doc.page_content}
    for doc in corpus
])
display(corpus_df)

## 1. Recuperación híbrida

### Señal sparse con BM25

BM25 puntúa coincidencias de términos teniendo en cuenta su frecuencia en cada documento, su rareza en el corpus y la longitud documental. `rank_bm25` deja visible la puntuación de todos los documentos, lo que facilita examinar el ranking sin ocultar el cálculo detrás de una abstracción.

La tokenización siguiente es deliberadamente sencilla: minúsculas y secuencias alfanuméricas. En un sistema real podrían añadirse lematización, tratamiento específico de identificadores y analizadores propios del idioma.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-záéíóúüñ0-9]+(?:-[a-záéíóúüñ0-9]+)?", text.lower())


tokenized_corpus = [tokenize(doc.page_content) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def rank_bm25(query):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1]
    return pd.DataFrame([
        {
            "id": corpus[index].metadata["id"],
            "score_sparse": float(scores[index]),
            "texto": corpus[index].page_content,
        }
        for index in order
    ])

In [ ]:
for query_id, query in queries.items():
    print(f"{query_id}: {query}")
    display(rank_bm25(query).head(4).round({"score_sparse": 3}))

### Señal dense con embeddings

El bi-encoder codifica consulta y documentos por separado. Los vectores documentales se calculan una vez y pueden indexarse; en cada búsqueda solo se codifica la consulta.

La implementación usa `OpenAIEmbeddings` de LangChain cuando existe `OPENAI_API_KEY`. En caso contrario utiliza el modelo local multilingüe `paraphrase-multilingual-MiniLM-L12-v2`, pequeño en comparación con los modelos de recuperación de gran escala. La interfaz común mantiene sin cambios el cálculo de similitud y las etapas posteriores.

In [ ]:
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))

if USE_OPENAI:
    from langchain_openai import OpenAIEmbeddings

    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    embedding_backend = "OpenAI · text-embedding-3-small"
else:
    from sentence_transformers import SentenceTransformer

    local_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    embedding_backend = "Local · paraphrase-multilingual-MiniLM-L12-v2"

print(f"Backend dense: {embedding_backend}")

In [ ]:
document_texts = [doc.page_content for doc in corpus]

if USE_OPENAI:
    document_embeddings = np.asarray(embedding_model.embed_documents(document_texts))
else:
    document_embeddings = local_model.encode(document_texts, convert_to_numpy=True)

document_embeddings = document_embeddings / np.linalg.norm(
    document_embeddings, axis=1, keepdims=True
)


def embed_query(query):
    if USE_OPENAI:
        vector = np.asarray(embedding_model.embed_query(query))
    else:
        vector = local_model.encode(query, convert_to_numpy=True)
    return vector / np.linalg.norm(vector)


def rank_dense(query):
    scores = document_embeddings @ embed_query(query)
    order = np.argsort(scores)[::-1]
    return pd.DataFrame([
        {
            "id": corpus[index].metadata["id"],
            "score_dense": float(scores[index]),
            "texto": corpus[index].page_content,
        }
        for index in order
    ])

In [ ]:
for query_id, query in queries.items():
    sparse_top = rank_bm25(query).head(4)["id"].tolist()
    dense_top = rank_dense(query).head(4)["id"].tolist()
    print(f"{query_id}: {query}")
    display(pd.DataFrame({
        "posición": range(1, 5),
        "sparse": sparse_top,
        "dense": dense_top,
    }))

In [ ]:
query = queries["Q2"]
score_comparison = (
    rank_bm25(query)[["id", "score_sparse"]]
    .merge(rank_dense(query)[["id", "score_dense"]], on="id")
    .merge(corpus_df[["id", "texto"]], on="id")
    .sort_values("score_dense", ascending=False)
)
print(query)
display(score_comparison.round(3))

Las puntuaciones anteriores no son directamente comparables: BM25 no está acotado a un intervalo fijo, mientras que la similitud coseno está entre −1 y 1. Además, su distribución depende de la consulta. Sumarlas sin calibración haría que la escala, y no solo la relevancia, determinara el resultado.

### Fusión basada en scores

Una opción sencilla es normalizar por consulta mediante min–max y combinar:

$$s_{hybrid}(d)=\alpha\,\hat{s}_{sparse}(d)+(1-\alpha)\,\hat{s}_{dense}(d)$$

`alpha` controla el peso léxico. Esta normalización es transparente, pero sensible a valores extremos y poco estable cuando todos los scores son casi iguales. En producción conviene calibrar con datos de validación.

In [ ]:
def minmax_normalize(values):
    values = np.asarray(values, dtype=float)
    value_range = values.max() - values.min()
    if np.isclose(value_range, 0):
        return np.zeros_like(values)
    return (values - values.min()) / value_range

In [ ]:
def score_fusion(query, alpha=0.5):
    sparse = rank_bm25(query)[["id", "score_sparse"]]
    dense = rank_dense(query)[["id", "score_dense"]]
    fused = sparse.merge(dense, on="id")
    fused["sparse_norm"] = minmax_normalize(fused["score_sparse"])
    fused["dense_norm"] = minmax_normalize(fused["score_dense"])
    fused["score_fusion"] = (
        alpha * fused["sparse_norm"] + (1 - alpha) * fused["dense_norm"]
    )
    return (
        fused.merge(corpus_df[["id", "texto"]], on="id")
        .sort_values("score_fusion", ascending=False)
        .reset_index(drop=True)
    )


query = queries["Q1"]
fusion_by_alpha = pd.concat([
    score_fusion(query, alpha=value).head(4).assign(alpha=value, posicion=lambda frame: range(1, len(frame) + 1))
    for value in (0.2, 0.5, 0.8)
])
display(fusion_by_alpha[["alpha", "posicion", "id", "score_fusion", "texto"]].round(3))

### Reciprocal Rank Fusion

RRF ignora las magnitudes y utiliza únicamente las posiciones. Para un documento $d$ y varios rankings $R$:

$$RRF(d)=\sum_{r\in R}\frac{1}{k+rank_r(d)}$$

El parámetro `k` suaviza la diferencia entre posiciones. RRF suele ser robusto cuando los recuperadores producen scores incompatibles. A cambio, descarta información sobre cuánta ventaja tenía un documento sobre otro dentro de cada ranking.

In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for position, document_id in enumerate(ranking, start=1):
            fused_scores[document_id] += 1 / (k + position)
    return sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)


def rrf_fusion(query, k=60):
    sparse_ids = rank_bm25(query)["id"].tolist()
    dense_ids = rank_dense(query)["id"].tolist()
    scores = reciprocal_rank_fusion([sparse_ids, dense_ids], k=k)
    result = pd.DataFrame(scores, columns=["id", "score_rrf"])
    return result.merge(corpus_df[["id", "texto"]], on="id")

In [ ]:
def compare_fusions(query, top_k=5):
    score_ids = score_fusion(query, alpha=0.5).head(top_k)["id"].tolist()
    rrf_ids = rrf_fusion(query).head(top_k)["id"].tolist()
    return pd.DataFrame({
        "posición": range(1, top_k + 1),
        "score fusion": score_ids,
        "RRF": rrf_ids,
    })


for query_id, query in queries.items():
    print(f"{query_id}: {query}")
    display(compare_fusions(query))

## 2. Retrieve-then-rerank

Un recuperador rápido genera un conjunto pequeño de candidatos y un modelo más expresivo los reordena:

1. **Retrieve:** BM25, bi-encoder dense o un híbrido buscan en todo el corpus.
2. **Rerank:** un cross-encoder recibe cada pareja `(consulta, documento)` y estima su relevancia conjunta.

El bi-encoder permite precomputar los documentos, pero comprime cada texto en un solo vector. El cross-encoder deja que todos los tokens de la consulta y del documento interactúen dentro del modelo; suele ordenar mejor, aunque requiere una inferencia por candidato. El reranker no examina documentos fuera del conjunto recuperado: puede mejorar el **orden**, pero no recuperar cobertura ya perdida.

In [ ]:
rerank_query = queries["Q4"]
top_k = 5
candidate_sources = pd.DataFrame({
    "posición": range(1, top_k + 1),
    "sparse": rank_bm25(rerank_query).head(top_k)["id"].tolist(),
    "dense": rank_dense(rerank_query).head(top_k)["id"].tolist(),
    "hybrid_RRF": rrf_fusion(rerank_query).head(top_k)["id"].tolist(),
})
display(candidate_sources)

In [ ]:
candidate_count = 6
candidates = rrf_fusion(rerank_query).head(candidate_count).copy()
candidates["posición_inicial"] = range(1, len(candidates) + 1)
display(candidates[["posición_inicial", "id", "score_rrf", "texto"]].round(4))

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
pairs = [(rerank_query, text) for text in candidates["texto"]]
candidates["score_reranker"] = reranker.predict(pairs)
reranked = candidates.sort_values("score_reranker", ascending=False).reset_index(drop=True)
reranked["posición_reranker"] = range(1, len(reranked) + 1)

In [ ]:
before_after = (
    candidates[["id", "posición_inicial", "texto"]]
    .merge(reranked[["id", "posición_reranker", "score_reranker"]], on="id")
    .sort_values("posición_reranker")
)
display(before_after.round({"score_reranker": 3}))

## 3. Late interaction y representaciones multi-vector

En un bi-encoder, todo el documento queda comprimido en un vector. Esa representación es eficiente, pero puede diluir detalles locales. Un modelo de interacción tardía como ColBERT conserva varios vectores, habitualmente uno por token contextualizado:

- consulta y documento se codifican por separado, por lo que los vectores documentales pueden precomputarse;
- la interacción ocurre después, durante el scoring;
- cada vector de la consulta busca su mejor coincidencia en el documento.

La regla MaxSim agrega esas mejores coincidencias:

$$score(q,d)=\sum_i \max_j sim(q_i,d_j)$$

Los vectores pequeños siguientes no proceden de un modelo entrenado; aíslan la mecánica de MaxSim sin exigir un índice ColBERT completo.

In [ ]:
query_tokens = ["coche", "eléctrico"]
query_vectors = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])

document_token_vectors = {
    "D_A": (
        ["automóvil", "batería", "ciudad"],
        np.array([[0.95, 0.05, 0.00], [0.05, 0.96, 0.00], [0.10, 0.05, 0.85]]),
    ),
    "D_B": (
        ["coche", "gasolina", "carretera"],
        np.array([[0.98, 0.02, 0.00], [0.10, 0.05, 0.90], [0.20, 0.05, 0.80]]),
    ),
}

In [ ]:
def cosine_matrix(left, right):
    left = left / np.linalg.norm(left, axis=1, keepdims=True)
    right = right / np.linalg.norm(right, axis=1, keepdims=True)
    return left @ right.T


def maxsim_score(query_matrix, document_matrix):
    similarities = cosine_matrix(query_matrix, document_matrix)
    maxima = similarities.max(axis=1)
    return similarities, maxima, maxima.sum()


maxsim_results = []
for document_id, (document_tokens, vectors) in document_token_vectors.items():
    matrix, maxima, score = maxsim_score(query_vectors, vectors)
    print(document_id)
    display(pd.DataFrame(matrix, index=query_tokens, columns=document_tokens).round(3))
    display(pd.DataFrame({"token_consulta": query_tokens, "mejor_similitud": maxima}).round(3))
    maxsim_results.append({"documento": document_id, "score_MaxSim": score})

display(pd.DataFrame(maxsim_results).sort_values("score_MaxSim", ascending=False).round(3))

In [ ]:
architecture_comparison = pd.DataFrame([
    {"arquitectura": "Bi-encoder", "representación documental": "un vector", "precomputable": "sí", "interacción": "coseno/dot product", "coste online": "bajo"},
    {"arquitectura": "Late interaction", "representación documental": "varios vectores", "precomputable": "sí", "interacción": "MaxSim", "coste online": "medio"},
    {"arquitectura": "Cross-encoder", "representación documental": "depende de la consulta", "precomputable": "no", "interacción": "atención conjunta", "coste online": "alto"},
])
display(architecture_comparison)

## 4. Recuperación multimodal

Un espacio multimodal compatible permite comparar elementos de distinta naturaleza. Un codificador de texto y otro de imagen, por ejemplo, se entrenan para que descripciones e imágenes relacionadas queden próximas. Audio y vídeo pueden añadirse mediante codificadores alineados con el mismo espacio.

Esto habilita consultas como texto → imagen, imagen → texto o audio → vídeo con la misma operación de similitud. La compatibilidad no surge de concatenar vectores arbitrarios: es consecuencia del entrenamiento contrastivo o de una alineación explícita entre modalidades.

El ejemplo siguiente es conceptual pero ejecutable. Los cuatro ejes representan rasgos semánticos compartidos —naturaleza, ciudad, sonido y movimiento— y no píxeles ni ondas reales.

In [ ]:
multimodal_items = pd.DataFrame([
    {"id": "M1", "modalidad": "imagen", "contenido": "bosque con un río", "vector": np.array([0.95, 0.05, 0.00, 0.20])},
    {"id": "M2", "modalidad": "audio", "contenido": "agua fluyendo y aves", "vector": np.array([0.82, 0.00, 0.65, 0.10])},
    {"id": "M3", "modalidad": "vídeo", "contenido": "tráfico en una avenida", "vector": np.array([0.05, 0.90, 0.55, 0.85])},
    {"id": "M4", "modalidad": "texto", "contenido": "sendero tranquilo en la naturaleza", "vector": np.array([0.90, 0.05, 0.05, 0.25])},
])
text_query_vector = np.array([0.92, 0.02, 0.20, 0.15])

item_matrix = np.vstack(multimodal_items["vector"])
item_matrix = item_matrix / np.linalg.norm(item_matrix, axis=1, keepdims=True)
normalized_query = text_query_vector / np.linalg.norm(text_query_vector)

cross_modal_results = multimodal_items.drop(columns="vector").copy()
cross_modal_results["similitud"] = item_matrix @ normalized_query
display(cross_modal_results.sort_values("similitud", ascending=False).round(3))

## 5. Decisiones arquitectónicas

| Método | Eficiencia de búsqueda | Precomputación documental | Interacción consulta-documento | Interpretabilidad | Coste típico |
|---|---:|---:|---|---|---:|
| Sparse / BM25 | Muy alta | Índice invertido | Coincidencia de términos | Alta | Bajo |
| Dense / bi-encoder | Alta con índice vectorial | Sí, un vector | Similitud global | Media-baja | Bajo–medio |
| Hybrid | Alta | Sí, dos índices | Fusión de señales | Media | Medio |
| Reranking / cross-encoder | Baja sobre todo el corpus; viable sobre Top-k | No para el score final | Atención conjunta | Media | Alto |
| Late interaction | Media-alta | Sí, varios vectores | MaxSim local | Media | Medio–alto |
| Multimodal | Depende del índice y codificadores | Habitualmente sí | Similitud en espacio alineado | Baja-media | Variable |

Una arquitectura habitual combina etapas: recuperación híbrida para cobertura, RRF cuando las escalas son incompatibles y un cross-encoder sobre pocos candidatos para precisión. Late interaction ofrece un punto intermedio entre la eficiencia del bi-encoder y la riqueza del cross-encoder. La recuperación multimodal extiende la misma lógica geométrica, siempre que los codificadores produzcan representaciones realmente compatibles.